In [1]:
# Run first on Google Colab
!pip install qutip -q

# §4 Shor's Factoring Algorithm

**Course:** Introductory Quantum Computing — Summer School

## Learning objectives
- Build the modular-multiplication unitary $U_{x,N}$ and verify its eigenstructure
- Run the full Shor pipeline for $N=15$, $x=7$ and recover factors via continued fractions
- Connect the quantum order-finding to the classical post-processing


In [2]:
import numpy as np
import qutip as qt
import matplotlib.pyplot as plt
from fractions import Fraction
from math import gcd

print(f"QuTiP {qt.__version__}")

# ── Helpers carried over from Notebook 1 ─────────────────────────────────────
def qft_matrix(n):
    N = 2**n
    omega = np.exp(2j * np.pi / N)
    F = np.array([[omega**(j*k) / np.sqrt(N) for k in range(N)] for j in range(N)])
    return qt.Qobj(F, dims=[[2]*n, [2]*n])

def apply_to_qubit(gate, n, qubit):
    ops = [qt.qeye(2)] * n
    ops[qubit] = gate
    return qt.tensor(ops)

def controlled_u(U, n, ctrl, tgt):
    P0 = qt.basis(2,0) * qt.basis(2,0).dag()
    P1 = qt.basis(2,1) * qt.basis(2,1).dag()
    ops0 = [qt.qeye(2)] * n;  ops0[ctrl] = P0
    ops1 = [qt.qeye(2)] * n;  ops1[ctrl] = P1;  ops1[tgt] = U
    return qt.tensor(ops0) + qt.tensor(ops1)

def swap_gate(n, i, j):
    N = 2**n
    mat = np.zeros((N, N), dtype=complex)
    for k in range(N):
        bits = list(format(k, f'0{n}b'))
        bits[i], bits[j] = bits[j], bits[i]
        mat[int(''.join(bits), 2), k] = 1.0
    return qt.Qobj(mat, dims=[[2]*n, [2]*n])

def qft_circuit(n):
    U = qt.tensor([qt.qeye(2)] * n)
    H1 = qt.gates.hadamard_transform(1)
    for i in range(n):
        U = apply_to_qubit(H1, n, i) * U
        for j in range(i+1, n):
            m = j - i + 1
            Rm = qt.Qobj(np.diag([1.0, np.exp(2j*np.pi/2**m)]))
            U = controlled_u(Rm, n, j, i) * U
    for i in range(n // 2):
        U = swap_gate(n, i, n-1-i) * U
    return U

def qpe(U_system, eigenstate, t_clock):
    sys_dim   = U_system.shape[0]
    sys_n     = int(np.round(np.log2(sys_dim)))
    sys_dims  = [[2]*sys_n, [2]*sys_n]

    zero = qt.basis(2, 0)
    H1   = qt.gates.hadamard_transform(1)
    I_sys = qt.qeye(sys_dim);  I_sys.dims = sys_dims

    clock_zero = qt.tensor([zero]*t_clock)
    psi = qt.tensor(clock_zero, eigenstate)

    Ht = qt.gates.hadamard_transform(t_clock)
    H_full = qt.tensor(Ht, I_sys)
    total_n = t_clock + sys_n
    H_full.dims = [[2]*total_n, [2]*total_n]
    psi = H_full * psi

    for j in range(t_clock):
        power = t_clock - 1 - j
        U_pow = U_system.copy()
        for _ in range(power):
            U_pow = U_pow * U_pow

        P0 = qt.basis(2,0) * qt.basis(2,0).dag()
        P1 = qt.basis(2,1) * qt.basis(2,1).dag()
        ops0 = [qt.qeye(2)]*t_clock;  ops0[j] = P0
        ops1 = [qt.qeye(2)]*t_clock;  ops1[j] = P1
        clock_P0 = qt.tensor(ops0)
        clock_P1 = qt.tensor(ops1)

        gate = qt.tensor(clock_P0, I_sys) + qt.tensor(clock_P1, U_pow)
        gate.dims = [[2]*total_n, [2]*total_n]
        psi = gate * psi

    QFT_dag = qft_circuit(t_clock).dag()
    QFT_full = qt.tensor(QFT_dag, I_sys)
    QFT_full.dims = [[2]*total_n, [2]*total_n]
    psi = QFT_full * psi

    return psi

def measure_clock(psi_out, t_clock, sys_n, n_shots=2048):
    N_clock = 2**t_clock
    N_sys   = 2**sys_n

    amps = psi_out.full().flatten()
    probs_clock = np.zeros(N_clock)
    for k in range(N_clock):
        for s in range(N_sys):
            probs_clock[k] += abs(amps[k * N_sys + s])**2

    outcomes = np.random.choice(N_clock, size=n_shots, p=probs_clock)
    counts = {k: int(np.sum(outcomes == k)) for k in range(N_clock) if np.sum(outcomes == k) > 0}
    return counts, probs_clock

print("QPE helpers defined.")


QuTiP 5.3.0
QPE helpers defined.


In [ ]:
#@title 🔒 Solution vault — enter the key from class to unlock  {display-mode: "form"}
# Solutions are obfuscated. Use the unlock key your instructor hands out in
# class, then run the relevant exercise's cell, e.g. reveal_solution("1.1", key="...").
import base64, zlib, hashlib

_VAULT = {
    "4.2": "Fmp0dkHrpGe7XbVUIJPuYv0UThnI7Fgo17PLQtZSpQyyKFCKPUjIcyU2eGUbJPNDeltwaDjFOj1kxyaIc9JDxard8aWZsWy5mkchhyRPWgyxm0yjto5X7TI/sKqeGGU7mcmIMDLMrOeBV1G/qDPU/t+Lmz7GhKpqbCWxeczgWxnyn8/EUiyFX8UDNlYbcmiIVQThJBjxajzwLiE22E6G8erAvR//Gppsr+jdspYkGDERPYGTcdZVGOWEEl8SI3Kh8yLWwWwQQ8P0d3gcqADxGwE5rM6Wd8q9N5XhTsAVlH/8Oy0BaN15fp1B3UzZQ1KL82OMAdg6JRVnlYT11bQE4El6LUKayyDcCtyN6BPrNxlfr3cLFkkbd5ibrT6B1dZ9UjjE+zJ0+Xy7vGEcjJdTy9+xxirJsBepnxGEeDZmW3jg/LQAUJS8QuwT5e+s4HAqC6loeo7xxxsN2+zUx1VINoY4gCR1JNEmZfVQNaesgVKlOlJjH207mTlpxDMPgbC9JMLu+W1or41uayvZ1y9L7nXeGc3cmwAoYDMUyib1hYvIvmMkE79P4uNLNyl4OLNkrRATbifRM3jrC9+TyzR2CHGEZ96Okz2N8uoM7R0Z6WnqK5kU2Muxpzmhnh5/8BlIczlce/Cd5VbdOORk+hqOJfZjSIuvpw8+9JiHK58Mpz8RTssjAbLuQTAJsGCeVtJPGSmVPYdNct6u87TC1Hc2JEnI2q2rl93Tm9ZZjRI5GU5g+HB8XdrCSExoeSX37yJWHtIcM6gLGDB5koXtIGJdyG/oa1ZO3WlIsmJIWRvkaliSFxinzqyIN62zPzhbdXg1USlMAO5Y1uLMJEiscmoDdTvMKf9/nhj/KBhXBRSJ1JkHcGvvpdT+/J919YvR+AZWW9zf2sdkOoV0UB0hLhyqfm12XbWtXh+dBlvC4T8CUL6visoBS664qKn7iTtPWoa2oRhu4wnjMAXsVG1K7e6dk5QNwWNIEo4sqWxEi8HWvYbW5gw7mti53JGYOVCs8tKGYCSN82+YCHcB0u3fDI4igsjW2CSeTdbU2J5ZKsG6u8SsBPWAWCkjLsloqymTnCNGlzrRhArGLx97DdbYesVuhNzsYbuF7jOeJYy/RC1uE2vnreJiMb3wq2QqvjKx1l0151yG9d3vnDbCAVh596uOSg/9aCfw9BkMiu/Php+cOsQlEi/iFrHzAlCHuUbhATTAWlkmU7p+4qSAu4Ml1wfYs2y2AvKVl3+4tVwDSz3T88nWxKrMe1/tSBlB6nNtv4nDigLjn0NR6WRO5haNmlP+BHcXVbTeDMskfuGa7idKaRy0FLZ5p2ApBlgDgbBEzOEqvox4uxb021BTwOVAI5SO3CK7tspngWupZOtHfd/s3/MRTtRqkcTdDv4uzjVkcrzeMGuuTsUP01zOxoeqzawAHAh4FaIqPUnEsmWxLsNQNNq9TUpPkGcimGxnXlA0jTmalA7TdvvEiOSBoinyviggRays71vAU0RH/frvnIrASu6MfiGAW57Nqo4KfQ+8q1LgqqmmsTh/mDLpW4KH",
}

def reveal_solution(exercise, key=""):
    """Print the solution for `exercise` (e.g. "1.1"), given the class unlock key."""
    blob = _VAULT.get(str(exercise))
    if blob is None:
        print(f"No solution stored for exercise {exercise!r}."); return
    kb = key.strip().encode()
    if not kb:
        print("Enter the key your instructor gave you, e.g. "
              f'reveal_solution("{exercise}", key="...").'); return
    data = base64.b64decode(blob.encode())
    out = bytes(b ^ kb[i % len(kb)] for i, b in enumerate(data))
    if out[:4] != hashlib.sha256(kb).digest()[:4]:
        print("✗ Wrong key — check the key your instructor gave you for THIS notebook."); return
    try:
        print(zlib.decompress(out[4:]).decode("utf-8"))
    except Exception:
        print("✗ Wrong key — check the key your instructor gave you for THIS notebook.")


---
## Part 2: Shor's Factoring Algorithm

### 4.1 Structure of the algorithm

**Goal:** Factor $N$ by finding the *order* $r = \mathrm{ord}_N(x)$ — the least $r > 0$ with $x^r \equiv 1 \pmod{N}$.

**Classical reduction (Proposition 4.1):** If $r$ is even and $x^{r/2} \not\equiv -1 \pmod{N}$,
then $\gcd(x^{r/2}\pm 1, N)$ are non-trivial factors.
A random $x$ satisfies the conditions with probability $\geq 1/2$ for semiprime $N=pq$.

**Quantum order-finding** (Algorithm 4.2):
1. Build $U_{x,N}|y\rangle = |xy\bmod N\rangle$ — a permutation unitary
2. Run QPE on $U_{x,N}$ with input $|1\rangle$ — the state $|1\rangle = \frac{1}{\sqrt{r}}\sum_s|u_s\rangle$
   is a uniform superposition over all eigenstates $|u_s\rangle$ with eigenphases $s/r$
3. Apply continued fractions to the QPE output to recover $r$

**Eigenstructure:** $U_{x,N}|u_s\rangle = e^{2\pi is/r}|u_s\rangle$ where
$|u_s\rangle = \frac{1}{\sqrt{r}}\sum_{k=0}^{r-1}e^{-2\pi isk/r}|x^k\bmod N\rangle$.


In [3]:
def build_U_xN(x, N):
    """
    Modular multiplication unitary: U|y> = |x*y mod N> for y < N,
                                    U|y> = |y>              for y >= N.
    Operates on d = ceil(log2(N)) qubits.
    """
    d = int(np.ceil(np.log2(N)))
    dim = 2**d
    mat = np.zeros((dim, dim), dtype=complex)
    for y in range(dim):
        if y < N:
            mat[(x * y) % N, y] = 1.0
        else:
            mat[y, y] = 1.0          # identity on states >= N
    U = qt.Qobj(mat, dims=[[2]*d, [2]*d])
    # Verify unitarity
    assert np.allclose((U.dag()*U).full(), np.eye(dim)), "U not unitary!"
    return U

# ── Test with x=7, N=15 (textbook example) ────────────────────────────────────
x, N = 7, 15
U_15 = build_U_xN(x, N)
print(f"U_{x},{N} is unitary: True")
print(f"Matrix dimension: {U_15.shape}")

# Verify U|1> = |7>, U|7> = |49 mod 15> = |4>
d = int(np.ceil(np.log2(N)))
ket1 = qt.basis(2**d, 1);  ket1.dims = [[2]*d,[1]*d]
ket7 = qt.basis(2**d, 7);  ket7.dims = [[2]*d,[1]*d]
ket4 = qt.basis(2**d, 4);  ket4.dims = [[2]*d,[1]*d]
print(f"U|1⟩ = |7⟩: {np.allclose((U_15*ket1).full(), ket7.full())}")
print(f"U|7⟩ = |4⟩: {np.allclose((U_15*ket7).full(), ket4.full())}")


U_7,15 is unitary: True
Matrix dimension: (16, 16)
U|1⟩ = |7⟩: True
U|7⟩ = |4⟩: True


In [4]:
# ── Verify eigenstructure: eigenphases should be s/r for s=0..r-1 ────────────
x, N = 7, 15
r_true = 4   # 7^4 = 2401 ≡ 1 (mod 15), so ord_15(7) = 4

evals, evecs = U_15.eigenstates()
phases = np.angle(evals) / (2*np.pi) % 1.0  # eigenphases in [0,1)
phases_sorted = sorted(phases)
print(f"Eigenphases of U_{{7,15}} (should include {{0, 1/4, 2/4, 3/4}}):")
unique = sorted(set(round(p,3) for p in phases_sorted))
print(unique)

expected = [s/r_true for s in range(r_true)]
print(f"Expected: {expected}")

# ── Build eigenstates manually ────────────────────────────────────────────────
def build_eigenstate(x, N, s, r, d):
    """Build |u_s> = (1/sqrt(r)) sum_k exp(-2pi*i*s*k/r) |x^k mod N>."""
    dim = 2**d
    vec = np.zeros(dim, dtype=complex)
    for k in range(r):
        y = pow(x, k, N)   # x^k mod N
        vec[y] += np.exp(-2j*np.pi*s*k/r) / np.sqrt(r)
    ket = qt.Qobj(vec, dims=[[2]*d, [1]*d])
    return ket

d = int(np.ceil(np.log2(N)))
for s in range(r_true):
    u_s = build_eigenstate(x, N, s, r_true, d)
    eigenval = (U_15 * u_s - np.exp(2j*np.pi*s/r_true) * u_s).norm()
    print(f"|u_{s}⟩: U|u_{s}⟩ = e^{{2πi·{s}/{r_true}}}|u_{s}⟩  error = {eigenval:.2e}")


Eigenphases of U_{7,15} (should include {0, 1/4, 2/4, 3/4}):
[np.float64(0.0), np.float64(0.25), np.float64(0.5), np.float64(0.75)]
Expected: [0.0, 0.25, 0.5, 0.75]
|u_0⟩: U|u_0⟩ = e^{2πi·0/4}|u_0⟩  error = 0.00e+00
|u_1⟩: U|u_1⟩ = e^{2πi·1/4}|u_1⟩  error = 1.22e-16
|u_2⟩: U|u_2⟩ = e^{2πi·2/4}|u_2⟩  error = 2.45e-16
|u_3⟩: U|u_3⟩ = e^{2πi·3/4}|u_3⟩  error = 3.67e-16


In [5]:
# ── Run quantum order finding: QPE on U_{7,15} with input |1> ────────────────
x, N = 7, 15
d = int(np.ceil(np.log2(N)))   # 4 system qubits
t = 2*d + 3                     # 11 clock qubits (per Algorithm 4.2)

U_xN = build_U_xN(x, N)

# Initial system state |1>
sys_init = qt.basis(2**d, 1);  sys_init.dims = [[2]*d, [1]*d]

print(f"Running QPE for U_{{{x},{N}}} with t={t} clock qubits...")
print("(This may take 10–20 seconds for t=11)")
psi_out = qpe(U_xN, sys_init, t)

_, probs = measure_clock(psi_out, t, d, n_shots=1)
print(f"QPE done. Non-negligible clock outcomes (P > 1/2^t):")
threshold = 0.5 / 2**t
top_k = sorted([k for k in range(2**t) if probs[k] > threshold], key=lambda k: -probs[k])
for k in top_k:
    print(f"  k={k:4d}  φ̃={k/2**t:.5f}  P={probs[k]:.4f}")
if not top_k:
    print("  (no outcomes above threshold — check QPE setup)")


Running QPE for U_{7,15} with t=11 clock qubits...
(This may take 10–20 seconds for t=11)
QPE done. Non-negligible clock outcomes (P > 1/2^t):
  k=   0  φ̃=0.00000  P=0.2500
  k= 512  φ̃=0.25000  P=0.2500
  k=1024  φ̃=0.50000  P=0.2500
  k=1536  φ̃=0.75000  P=0.2500


In [6]:
# ── Continued fractions: recover r from φ̃ ≈ s/r ─────────────────────────────
def continued_fraction(phi_est, N):
    """
    Return the denominator q of the best rational approximation p/q to phi_est
    with q <= N (using Python's Fraction with limit_denominator).
    """
    frac = Fraction(phi_est).limit_denominator(N)
    return frac.numerator, frac.denominator

# Test on the known eigenphases of U_{7,15}
print("Continued fractions applied to QPE outputs for U_{7,15}:")
print(f"{'k':>6}  {'φ̃':>8}  {'s/r':>8}  {'r candidate':>12}  {'checks out':>12}")
for k in [512, 1024, 1536, 0]:      # expected outputs: s/r ∈ {1/4, 2/4, 3/4, 0/4}
    phi_est = k / 2**t
    s_est, r_est = continued_fraction(phi_est, N)
    checks = (pow(x, r_est, N) == 1) and (r_est > 0)
    print(f"{k:>6}  {phi_est:>8.5f}  {s_est}/{r_est:>4}  {r_est:>12}  {checks!s:>12}")

# ── Full classical post-processing ────────────────────────────────────────────
def shor_postprocess(x, N, r):
    """Try to extract factors from order r."""
    if r % 2 != 0:
        return None, "r is odd"
    half = pow(x, r // 2, N)
    if half == N - 1:
        return None, "x^{r/2} ≡ -1 (mod N)"
    p = gcd(half + 1, N)
    q = gcd(half - 1, N)
    factors = [f for f in [p, q] if 1 < f < N]
    return factors, "ok"

r = 4   # known order
factors, msg = shor_postprocess(x, N, r)
print(f"\nFactoring N={N} with x={x}, r={r}: factors = {factors}")
print(f"Check: {factors[0]} × {N//factors[0]} = {factors[0]*(N//factors[0])}")


Continued fractions applied to QPE outputs for U_{7,15}:
     k        φ̃       s/r   r candidate    checks out
   512   0.25000  1/   4             4          True
  1024   0.50000  1/   2             2         False
  1536   0.75000  3/   4             4          True
     0   0.00000  0/   1             1         False

Factoring N=15 with x=7, r=4: factors = [5, 3]
Check: 5 × 3 = 15


### Exercise 4.2 — Factor $N=91$ with $x=4$

Run the **full quantum order-finding pipeline** — don't use the classically-known order in the factoring step.

**(a)** Compute $\mathrm{ord}_{91}(4)$ classically (smallest $r$ with $4^r \equiv 1 \pmod{91}$) as a reference.

**(b)** Verify the eigenphases of $U_{4,91}$ match $\{0, 1/r, 2/r, \ldots\}$.

**(c)** Run QPE on $U_{4,91}$ with input $|1\rangle$ (use $t=10$ clock qubits; $\sim$5 s).
Apply continued fractions to the **measured** phases to recover $r$, then factor $91$.

*Note:* $N=91$ needs $d=\lceil\log_2 91\rceil = 7$ system qubits. Full precision would need
$t=2d+1=15$ clock qubits (22 total — infeasible to simulate densely). With $t=10$ the order
$r=6$ is small enough that the continued-fraction step still recovers it reliably.

In [ ]:
# YOUR CODE HERE
from math import lcm

N_ex, x_ex = 91, 4
d_ex = int(np.ceil(np.log2(N_ex)))   # 7 system qubits

# (a) Reference order: smallest r with x^r = 1 mod N
# r_ref = ...

# (b) Build U_{4,91}, check its eigenphases against {s/r}

# (c) Run QPE with input |1>, t=10 clock qubits:
#   psi_out = qpe(U_ex, sys_init, t=10)
#   _, probs = measure_clock(psi_out, 10, d_ex, n_shots=1)
# For each significant outcome k, get denominator via continued_fraction(k/2^t, N).
# Recover r as the lcm of those denominators, then call shor_postprocess(x_ex, N_ex, r).

In [ ]:
#@title 🔒 Exercise 4.2 (locked)  {display-mode: "form"}
reveal_solution("4.2", key="PASTE-KEY-HERE")


---
## Summary

| Topic | Key result |
|-------|-----------|
| Shor reduction | Factor $N$ by finding $r = \mathrm{ord}_N(x)$; if $r$ even and $x^{r/2}\not\equiv -1$, $\gcd(x^{r/2}\pm 1,N)$ gives factors |
| $U_{x,N}$ eigenstructure | Eigenphases $s/r$; $|1\rangle = \frac{1}{\sqrt{r}}\sum_s|u_s\rangle$ — QPE on $|1\rangle$ samples $s/r$ uniformly |
| Continued fractions | Recover $r$ from $\tilde\varphi \approx s/r$; $O(\log^3 N)$ classical post-processing |
| Complexity | $O(n^3)$ gates ($n=\lceil\log_2 N\rceil$); exponential improvement over best known classical algorithm |

**Next:** Notebook 5 covers Grover's algorithm and amplitude estimation (§5).